# 宏观因子资产配置框架改进 - 复现研究

本Notebook复现国泰君安研报《大类资产配置量化模型研究系列之七：宏观因子资产配置框架的改进》

**研究目标**: 
1. 基于6大宏观因子（增长、通胀、利率、信用、汇率、流动性）构建资产配置框架
2. 改进宏观因子生成方法（波动率倒数加权、黄金组合汇率因子）
3. 改进因子暴露计算（带半衰期的多元线性回归）
4. 设置宏观观点打分规则

**注意**: 本项目需要tushare Pro账号获取完整数据。

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')

## 1. 导入项目模块

In [ ]:
from source import (
    DataLoader,
    FactorGenerator,
    FactorExposure,
    PortfolioOptimizer,
    MacroScoring,
    Backtest,
    Visualizer,
    ASSETS_CONFIG,
    FACTOR_CONFIG,
    BACKTEST_CONFIG,
    RESULTS_DIR,
    DATA_DIR,
)

print('All modules imported successfully')

## 2. 配置参数

In [ ]:
START_DATE = BACKTEST_CONFIG['start_date']
END_DATE = BACKTEST_CONFIG['end_date']
INITIAL_CAPITAL = BACKTEST_CONFIG['initial_capital']

print(f'回测期间: {START_DATE} 至 {END_DATE}')
print(f'初始资金: {INITIAL_CAPITAL:,.2f}')
print(f'\n配置资产数量: {len(ASSETS_CONFIG)}')
print(f'配置因子数量: {len(FACTOR_CONFIG)}')

## 3. 数据加载

In [ ]:
data_loader = DataLoader(use_cache=True)
print('数据加载器初始化完成')

print('\n配置的交易资产:')
for key, info in ASSETS_CONFIG.items():
    print(f'  {key}: {info["name"]} ({info["code"]})')

## 4. 生成宏观因子

In [ ]:
factor_generator = FactorGenerator(data_loader)

print('正在生成宏观因子...')
print('注意: 完整因子生成需要tushare Pro API连接')

factors = factor_generator.generate_all_factors(START_DATE, END_DATE)

print(f'\n成功生成因子数量: {len(factors)}')
for name, df in factors.items():
    if len(df) > 0:
        print(f'  {name}: {len(df)} 条记录')

## 5. 加载资产收益率数据

In [ ]:
asset_list = list(ASSETS_CONFIG.keys())

print('正在加载资产收益率数据...')
asset_returns = data_loader.load_asset_returns(
    asset_list, START_DATE, END_DATE
)

if len(asset_returns) > 0:
    print(f'\n成功加载资产收益数据')
    print(f'数据时间范围: {asset_returns.index.min()} 至 {asset_returns.index.max()}')
    print(f'数据条数: {len(asset_returns)}')
    print(f'\n资产列表: {list(asset_returns.columns)}')
else:
    print('\n警告: 未能获取资产数据，请检查tushare API连接')

## 6. 计算因子暴露矩阵

In [ ]:
factor_exposure = FactorExposure(data_loader, factor_generator)

print('正在计算因子暴露矩阵...')
print('使用改进后的方法: 带半衰期的多元线性回归（窗口期5年，半衰期1年）')

high_freq_factors = factor_generator.get_high_freq_factors(START_DATE, END_DATE)

if len(high_freq_factors) > 0 and len(asset_returns) > 0:
    exposure_matrix = factor_exposure.get_exposure_matrix_with_r_squared(
        asset_returns, high_freq_factors
    )
    
    if not exposure_matrix.empty:
        print('\n因子暴露矩阵:')
        print(exposure_matrix.round(4))
    else:
        print('\n警告: 因子暴露矩阵为空')

## 7. 运行回测

In [ ]:
backtest = Backtest(
    start_date=START_DATE,
    end_date=END_DATE,
    initial_capital=INITIAL_CAPITAL,
    commission_rate=BACKTEST_CONFIG['commission_rate'],
    stamp_tax=BACKTEST_CONFIG['stamp_tax'],
    rebalance_freq='monthly',
)

print('='*60)
print('运行策略回测...')
print('='*60)

results = backtest.run_backtest(use_macro_view=False)

## 8. 回测结果分析

In [ ]:
if results:
    print('\n' + '='*60)
    print('回测绩效指标')
    print('='*60)
    
    metrics = {
        '总收益率': f"{results.get('total_return', 0)*100:.2f}%",
        '年化收益率': f"{results.get('annualized_return', 0)*100:.2f}%",
        '年化波动率': f"{results.get('annualized_volatility', 0)*100:.2f}%",
        '夏普比率': f"{results.get('sharpe_ratio', 0):.2f}",
        '最大回撤': f"{results.get('max_drawdown', 0)*100:.2f}%",
        '胜率': f"{results.get('win_rate', 0)*100:.2f}%",
        '盈亏比': f"{results.get('profit_loss_ratio', 0):.2f}",
        '交易次数': f"{results.get('num_trades', 0)}",
    }
    
    for name, value in metrics.items():
        print(f'{name}: {value}')

## 9. 运行基准回测

In [ ]:
benchmark_values = backtest.run_benchmark_backtest()

if len(benchmark_values) > 0:
    benchmark_return = (benchmark_values.iloc[-1] / benchmark_values.iloc[0]) - 1
    print(f'\n基准（等权）总收益率: {benchmark_return*100:.2f}%')

## 10. 可视化结果

In [ ]:
visualizer = Visualizer(output_dir=str(RESULTS_DIR))

if results and 'portfolio_values' in results:
    portfolio_values = results['portfolio_values']
    weights_history = results.get('weights_history')
    
    print('生成可视化图表...')
    
    visualizer.generate_backtest_report(
        backtest_results=results,
        portfolio_values=portfolio_values,
        benchmark_values=benchmark_values,
        weights_history=weights_history,
        exposure_matrix=exposure_matrix if not exposure_matrix.empty else None,
        save_dir=str(RESULTS_DIR),
    )

## 11. 保存结果

In [ ]:
output_results_path = RESULTS_DIR / 'backtest_results.csv'
backtest.save_results(str(output_results_path))

if results and 'portfolio_values' in results:
    values_path = RESULTS_DIR / 'portfolio_values.csv'
    results['portfolio_values'].to_csv(values_path)
    print(f'组合净值序列已保存至: {values_path}')
    
    weights_path = RESULTS_DIR / 'weights_history.csv'
    results['weights_history'].to_csv(weights_path)
    print(f'权重历史已保存至: {weights_path}')

## 研报复现要点总结

### 1. 宏观因子体系
- **增长因子(Growth)**: PMI同比、固定资产投资、社消、进出口加权
- **通胀因子(Inflation)**: CPI、PPI加权
- **利率因子(IntRate)**: 10年期国债收益率
- **信用因子(Credit)**: 信用利差（AA中短票-国开债）
- **汇率因子(ExchRate)**: 美元兑人民币
- **流动性因子(Liquidity)**: M2同比-社融同比

### 2. 改进1：高频因子生成
- 使用波动率倒数加权合成高频增长和通胀因子
- 使用黄金组合（沪金- COMEX黄金）代替美元指数构造汇率因子

### 3. 改进2：因子暴露计算
- 将Lasso回归改为带半衰期的多远线性回归
- 回归窗口期从10年缩短至5年
- 半衰期设为1年

### 4. 改进3：宏观打分规则
- 将宏观观点分为5档（+2, +1, 0, -1, -2）
- 每档对应不同调整系数（1倍/0.5倍标准差）
- 目标因子暴露 = 基准因子暴露 + 调整系数 × 历史标准差

### 数据说明
本项目需要以下数据源支持:
- **tushare Pro**: 股票、债券、期货、宏观数据
- **yfinance**: COMEX黄金等国际商品数据

如遇数据获取问题，请检查:
1. tushare token是否有效
2. API连接是否正常
3. 网络是否能访问数据源